In [1]:
import os
import json
import time
import warnings
import threading
import joblib

import numpy as np
import pandas as pd

import tkinter as tk
from tkinter import ttk, messagebox, filedialog, scrolledtext

from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from tabpfn import TabPFNRegressor

warnings.filterwarnings("ignore")


# ================================================================
# CHLORIDE INGRESS PREDICTOR
# FAST + CLEAN + AESTHETIC GUI
# ================================================================

class ChlorideIngressPredictor:

    # ============================================================
    # INITIALIZATION
    # ============================================================

    def __init__(self):

        self.setup_style()
        self.setup_paths()

        # --------------------------------------------------------
        # MODEL CONFIGURATION
        # --------------------------------------------------------

        self.TEST_SIZE = 0.20
        self.RANDOM_STATE = 42

        # Fast TabPFN configuration
        self.N_ESTIMATORS = 4

        self.model = None
        self.prediction_history = []

        # --------------------------------------------------------
        # DATA
        # --------------------------------------------------------

        self.load_and_preprocess_data()

        # --------------------------------------------------------
        # MODEL
        # --------------------------------------------------------

        self.load_or_train_model()

        # --------------------------------------------------------
        # GUI
        # --------------------------------------------------------

        self.create_gui()

        self.load_history()

    # ============================================================
    # STYLE
    # ============================================================

    def setup_style(self):

        self.colors = {

            "navy": "#17324D",
            "blue": "#2878B5",
            "light_blue": "#EAF3F9",

            "teal": "#1F8A8A",
            "light_teal": "#E8F6F6",

            "green": "#2E8B57",
            "light_green": "#EAF6EF",

            "orange": "#E67E22",
            "light_orange": "#FFF3E8",

            "red": "#C0392B",
            "light_red": "#FCECEA",

            "purple": "#6C5CE7",
            "light_purple": "#F0EEFF",

            "background": "#F4F7FA",
            "card": "#FFFFFF",
            "border": "#D9E1E8",

            "text": "#243447",
            "muted": "#6B7785",

            "white": "#FFFFFF"
        }

    # ============================================================
    # PATHS
    # ============================================================

    def setup_paths(self):

        self.data_path = (
            r"D:\2026 Work\My Papers\Chloride Ingress"
            r"\Data\Data.csv"
        )

        self.save_dir = (
            r"D:\2026 Work\Chloride Ingress"
            r"\Results\GUI_Application"
        )

        os.makedirs(
            self.save_dir,
            exist_ok=True
        )

        self.model_path = os.path.join(
            self.save_dir,
            "tabpfn_chloride_model.pkl"
        )

        self.preprocessor_path = os.path.join(
            self.save_dir,
            "chloride_preprocessor.pkl"
        )

        self.metrics_path = os.path.join(
            self.save_dir,
            "model_metrics.json"
        )

        self.history_file = os.path.join(
            self.save_dir,
            "prediction_history.json"
        )

    # ============================================================
    # DATA LOADING
    # ============================================================

    def load_and_preprocess_data(self):

        start = time.time()

        print("\n" + "=" * 70)
        print("LOADING CHLORIDE INGRESS DATA")
        print("=" * 70)

        encodings = [
            "utf-8",
            "utf-8-sig",
            "cp1252",
            "latin1"
        ]

        self.df = None

        for encoding in encodings:

            try:

                self.df = pd.read_csv(
                    self.data_path,
                    encoding=encoding
                )

                print(
                    f"CSV encoding: {encoding}"
                )

                break

            except UnicodeDecodeError:

                continue

        if self.df is None:

            raise ValueError(
                "Unable to decode the CSV file."
            )

        self.df.columns = (
            self.df.columns
            .astype(str)
            .str.strip()
        )

        print(
            f"Dataset shape: {self.df.shape}"
        )

        # --------------------------------------------------------
        # TARGET
        # --------------------------------------------------------

        target_col = (
            "Total chloride, % of binder mass"
        )

        if target_col not in self.df.columns:

            matches = [
                c for c in self.df.columns
                if "chloride" in c.lower()
            ]

            if matches:

                target_col = matches[0]

            else:

                raise ValueError(
                    "Target column was not found."
                )

        self.target_name = target_col

        print(
            f"Target: {self.target_name}"
        )

        # --------------------------------------------------------
        # TARGET CLEANING
        # --------------------------------------------------------

        self.df[self.target_name] = pd.to_numeric(
            self.df[self.target_name],
            errors="coerce"
        )

        self.df = self.df.dropna(
            subset=[self.target_name]
        )

        self.df = (
            self.df
            .drop_duplicates()
            .reset_index(drop=True)
        )

        # --------------------------------------------------------
        # EMPTY COLUMNS
        # --------------------------------------------------------

        empty_columns = [
            c for c in self.df.columns
            if self.df[c].isna().all()
        ]

        if empty_columns:

            self.df = self.df.drop(
                columns=empty_columns
            )

        # --------------------------------------------------------
        # X / Y
        # --------------------------------------------------------

        self.X_full = self.df.drop(
            columns=[self.target_name]
        ).copy()

        self.y_full = self.df[
            self.target_name
        ].copy()

        self.feature_names = (
            self.X_full.columns.tolist()
        )

        # --------------------------------------------------------
        # CATEGORICAL FEATURES
        # --------------------------------------------------------

        self.categorical_features = []

        for feature in self.feature_names:

            dtype = self.X_full[
                feature
            ].dtype

            if (
                dtype == "object"
                or
                str(dtype) == "category"
            ):

                self.categorical_features.append(
                    feature
                )

        print(
            f"Features: {len(self.feature_names)}"
        )

        print(
            f"Categorical features: "
            f"{len(self.categorical_features)}"
        )

        # --------------------------------------------------------
        # TRAIN TEST SPLIT
        # --------------------------------------------------------

        (
            self.X_train_raw,
            self.X_test_raw,
            self.y_train,
            self.y_test
        ) = train_test_split(
            self.X_full,
            self.y_full,
            test_size=self.TEST_SIZE,
            random_state=self.RANDOM_STATE
        )

        # --------------------------------------------------------
        # PREPROCESSING
        # --------------------------------------------------------

        self.feature_stats = {}
        self.categorical_maps = {}

        self.X_train = self.X_train_raw.copy()
        self.X_test = self.X_test_raw.copy()

        for feature in self.feature_names:

            # ====================================================
            # CATEGORICAL
            # ====================================================

            if feature in self.categorical_features:

                train_values = (
                    self.X_train[
                        feature
                    ]
                    .fillna("Missing")
                    .astype(str)
                )

                categories = sorted(
                    train_values.unique()
                )

                if "Missing" not in categories:

                    categories.append(
                        "Missing"
                    )

                category_to_code = {
                    category: index
                    for index, category
                    in enumerate(categories)
                }

                self.categorical_maps[
                    feature
                ] = category_to_code

                mode_values = (
                    train_values.mode()
                )

                if len(mode_values) > 0:

                    mode = mode_values.iloc[0]

                else:

                    mode = categories[0]

                self.feature_stats[
                    feature
                ] = {

                    "type": "categorical",

                    "categories": categories,

                    "mode": mode
                }

                self.X_train[
                    feature
                ] = (
                    train_values
                    .map(category_to_code)
                    .fillna(
                        category_to_code[
                            mode
                        ]
                    )
                    .astype(int)
                )

                test_values = (
                    self.X_test[
                        feature
                    ]
                    .fillna("Missing")
                    .astype(str)
                )

                self.X_test[
                    feature
                ] = (
                    test_values
                    .map(category_to_code)
                    .fillna(
                        category_to_code[
                            mode
                        ]
                    )
                    .astype(int)
                )

            # ====================================================
            # NUMERIC
            # ====================================================

            else:

                train_numeric = pd.to_numeric(
                    self.X_train[
                        feature
                    ],
                    errors="coerce"
                )

                train_numeric = (
                    train_numeric
                    .replace(
                        [np.inf, -np.inf],
                        np.nan
                    )
                )

                median_value = (
                    train_numeric.median()
                )

                mean_value = (
                    train_numeric.mean()
                )

                min_value = (
                    train_numeric.min()
                )

                max_value = (
                    train_numeric.max()
                )

                std_value = (
                    train_numeric.std()
                )

                self.feature_stats[
                    feature
                ] = {

                    "type": "numeric",

                    "mean": float(
                        mean_value
                    ),

                    "median": float(
                        median_value
                    ),

                    "min": float(
                        min_value
                    ),

                    "max": float(
                        max_value
                    ),

                    "std": float(
                        std_value
                    )
                }

                self.X_train[
                    feature
                ] = (
                    train_numeric
                    .fillna(median_value)
                )

                test_numeric = pd.to_numeric(
                    self.X_test[
                        feature
                    ],
                    errors="coerce"
                )

                test_numeric = (
                    test_numeric
                    .replace(
                        [np.inf, -np.inf],
                        np.nan
                    )
                    .fillna(median_value)
                )

                self.X_test[
                    feature
                ] = test_numeric

        # --------------------------------------------------------
        # TARGET STATISTICS
        # --------------------------------------------------------

        self.target_stats = {

            "min": float(
                self.y_train.min()
            ),

            "max": float(
                self.y_train.max()
            ),

            "mean": float(
                self.y_train.mean()
            ),

            "std": float(
                self.y_train.std()
            ),

            "median": float(
                self.y_train.median()
            )
        }

        # --------------------------------------------------------
        # SAVE PREPROCESSOR
        # --------------------------------------------------------

        joblib.dump(
            {
                "feature_names":
                    self.feature_names,

                "categorical_features":
                    self.categorical_features,

                "categorical_maps":
                    self.categorical_maps,

                "feature_stats":
                    self.feature_stats,

                "target_stats":
                    self.target_stats
            },
            self.preprocessor_path
        )

        print(
            f"Preprocessing completed in "
            f"{time.time() - start:.2f} s"
        )

    # ============================================================
    # DEVICE
    # ============================================================

    def get_device(self):

        try:

            import torch

            if torch.cuda.is_available():

                print(
                    "GPU detected."
                )

                return "cuda"

            print(
                "GPU not detected. CPU mode."
            )

            return "cpu"

        except Exception:

            return "cpu"

    # ============================================================
    # CATEGORICAL INDICES
    # ============================================================

    def get_categorical_indices(self):

        return [
            self.feature_names.index(feature)
            for feature
            in self.categorical_features
        ]

    # ============================================================
    # MODEL
    # ============================================================

    def load_or_train_model(self):

        start = time.time()

        self.device = self.get_device()

        # --------------------------------------------------------
        # LOAD EXISTING MODEL
        # --------------------------------------------------------

        if os.path.exists(
            self.model_path
        ):

            try:

                print(
                    "\nLoading saved TabPFN model..."
                )

                self.model = joblib.load(
                    self.model_path
                )

                self.calculate_test_metrics()

                print(
                    "Saved model loaded."
                )

                return

            except Exception as e:

                print(
                    f"Model loading failed: {e}"
                )

                self.model = None

        # --------------------------------------------------------
        # TRAIN
        # --------------------------------------------------------

        print("\n" + "=" * 70)
        print("TRAINING TABPFN MODEL")
        print("=" * 70)

        categorical_indices = (
            self.get_categorical_indices()
        )

        model_kwargs = {

            "device":
                self.device,

            "random_state":
                self.RANDOM_STATE,

            "n_estimators":
                self.N_ESTIMATORS
        }

        if categorical_indices:

            model_kwargs[
                "categorical_features_indices"
            ] = categorical_indices

        self.model = TabPFNRegressor(
            **model_kwargs
        )

        train_start = time.time()

        self.model.fit(
            self.X_train,
            self.y_train
        )

        train_time = (
            time.time() - train_start
        )

        print(
            f"Training time: "
            f"{train_time:.2f} s"
        )

        self.calculate_test_metrics()

        # --------------------------------------------------------
        # SAVE
        # --------------------------------------------------------

        joblib.dump(
            self.model,
            self.model_path
        )

        print(
            f"Model saved:\n"
            f"{self.model_path}"
        )

        print(
            f"Initialization time: "
            f"{time.time() - start:.2f} s"
        )

    # ============================================================
    # TEST METRICS
    # ============================================================

    def calculate_test_metrics(self):

        self.y_test_pred = (
            self.model.predict(
                self.X_test
            )
        )

        self.test_r2 = r2_score(
            self.y_test,
            self.y_test_pred
        )

        self.test_rmse = np.sqrt(
            mean_squared_error(
                self.y_test,
                self.y_test_pred
            )
        )

        self.test_mae = mean_absolute_error(
            self.y_test,
            self.y_test_pred
        )

        nonzero = (
            self.y_test.values != 0
        )

        if np.any(nonzero):

            self.test_mape = (
                np.mean(
                    np.abs(
                        (
                            self.y_test.values[
                                nonzero
                            ]
                            -
                            self.y_test_pred[
                                nonzero
                            ]
                        )
                        /
                        self.y_test.values[
                            nonzero
                        ]
                    )
                )
                * 100
            )

        else:

            self.test_mape = np.nan

        metrics = {

            "model":
                "TabPFNRegressor",

            "device":
                self.device,

            "n_estimators":
                self.N_ESTIMATORS,

            "random_state":
                self.RANDOM_STATE,

            "test_size":
                self.TEST_SIZE,

            "training_samples":
                len(self.X_train),

            "test_samples":
                len(self.X_test),

            "features":
                len(self.feature_names),

            "test_R2":
                float(self.test_r2),

            "test_RMSE":
                float(self.test_rmse),

            "test_MAE":
                float(self.test_mae),

            "test_MAPE_percent":
                float(self.test_mape)
                if np.isfinite(
                    self.test_mape
                )
                else None
        }

        with open(
            self.metrics_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                metrics,
                f,
                indent=4
            )

    # ============================================================
    # GUI
    # ============================================================

    def create_gui(self):

        self.root = tk.Tk()

        self.root.title(
            "Chloride Ingress Predictor | TabPFN"
        )

        self.root.geometry(
            "1450x900"
        )

        self.root.minsize(
            1150,
            750
        )

        self.root.configure(
            bg=self.colors["background"]
        )

        self.configure_ttk()

        # --------------------------------------------------------
        # HEADER
        # --------------------------------------------------------

        header = tk.Frame(
            self.root,
            bg=self.colors["navy"],
            height=90
        )

        header.pack(
            fill="x"
        )

        header.pack_propagate(
            False
        )

        title_frame = tk.Frame(
            header,
            bg=self.colors["navy"]
        )

        title_frame.pack(
            side="left",
            padx=28,
            pady=12
        )

        tk.Label(
            title_frame,
            text="CHLORIDE INGRESS PREDICTOR",
            font=("Segoe UI", 21, "bold"),
            fg=self.colors["white"],
            bg=self.colors["navy"]
        ).pack(
            anchor="w"
        )

        tk.Label(
            title_frame,
            text="TabPFN Regression Application",
            font=("Segoe UI", 10),
            fg="#D5E5F2",
            bg=self.colors["navy"]
        ).pack(
            anchor="w",
            pady=(3, 0)
        )

        model_badge = tk.Label(
            header,
            text="TABPFN",
            font=("Segoe UI", 10, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["white"],
            padx=15,
            pady=7
        )

        model_badge.pack(
            side="right",
            padx=28
        )

        # --------------------------------------------------------
        # NOTEBOOK
        # --------------------------------------------------------

        self.notebook = ttk.Notebook(
            self.root
        )

        self.notebook.pack(
            fill="both",
            expand=True,
            padx=14,
            pady=12
        )

        self.prediction_tab = (
            self.create_prediction_tab()
        )

        self.analysis_tab = (
            self.create_analysis_tab()
        )

        self.history_tab = (
            self.create_history_tab()
        )

        self.notebook.add(
            self.prediction_tab,
            text="  Prediction  "
        )

        self.notebook.add(
            self.analysis_tab,
            text="  Analysis  "
        )

        self.notebook.add(
            self.history_tab,
            text="  History  "
        )

        # --------------------------------------------------------
        # STATUS BAR
        # --------------------------------------------------------

        status_frame = tk.Frame(
            self.root,
            bg=self.colors["navy"],
            height=32
        )

        status_frame.pack(
            fill="x"
        )

        status_frame.pack_propagate(
            False
        )

        self.status_var = tk.StringVar(
            value="Model ready"
        )

        tk.Label(
            status_frame,
            textvariable=self.status_var,
            font=("Segoe UI", 9),
            fg=self.colors["white"],
            bg=self.colors["navy"]
        ).pack(
            side="left",
            padx=15
        )

        # Removed the R² display from status bar

        self.root.protocol(
            "WM_DELETE_WINDOW",
            self.on_closing
        )

    # ============================================================
    # TTK STYLE
    # ============================================================

    def configure_ttk(self):

        style = ttk.Style()

        try:

            style.theme_use(
                "clam"
            )

        except Exception:

            pass

        style.configure(
            ".",
            font=("Segoe UI", 10)
        )

        style.configure(
            "TNotebook",
            background=self.colors["background"],
            borderwidth=0
        )

        style.configure(
            "TNotebook.Tab",
            padding=(20, 10),
            font=("Segoe UI", 10, "bold")
        )

        style.configure(
            "TFrame",
            background=self.colors["background"]
        )

        style.configure(
            "TLabelframe",
            background=self.colors["background"]
        )

        style.configure(
            "TLabelframe.Label",
            font=("Segoe UI", 10, "bold"),
            foreground=self.colors["navy"]
        )

        style.configure(
            "TButton",
            font=("Segoe UI", 10, "bold"),
            padding=(12, 8)
        )

        style.configure(
            "Accent.TButton",
            foreground=self.colors["white"],
            background=self.colors["blue"],
            font=("Segoe UI", 11, "bold"),
            padding=(18, 10)
        )

        style.map(
            "Accent.TButton",
            background=[
                (
                    "active",
                    self.colors["navy"]
                )
            ]
        )

        style.configure(
            "TEntry",
            padding=7
        )

        style.configure(
            "TCombobox",
            padding=6
        )

    # ============================================================
    # PREDICTION TAB
    # ============================================================

    def create_prediction_tab(self):

        tab = ttk.Frame(
            self.notebook
        )

        main = tk.Frame(
            tab,
            bg=self.colors["background"]
        )

        main.pack(
            fill="both",
            expand=True,
            padx=12,
            pady=12
        )

        # --------------------------------------------------------
        # LEFT PANEL
        # --------------------------------------------------------

        left_card = tk.Frame(
            main,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        left_card.pack(
            side="left",
            fill="both",
            expand=True,
            padx=(0, 8)
        )

        tk.Label(
            left_card,
            text="Input Parameters",
            font=("Segoe UI", 15, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["card"]
        ).pack(
            anchor="w",
            padx=20,
            pady=(18, 2)
        )

        tk.Label(
            left_card,
            text="Enter the mixture and exposure parameters.",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["card"]
        ).pack(
            anchor="w",
            padx=20,
            pady=(0, 12)
        )

        # --------------------------------------------------------
        # SCROLL AREA
        # --------------------------------------------------------

        canvas = tk.Canvas(
            left_card,
            bg=self.colors["card"],
            highlightthickness=0
        )

        scrollbar = ttk.Scrollbar(
            left_card,
            orient="vertical",
            command=canvas.yview
        )

        input_frame = tk.Frame(
            canvas,
            bg=self.colors["card"]
        )

        input_frame.bind(
            "<Configure>",
            lambda e:
            canvas.configure(
                scrollregion=
                canvas.bbox("all")
            )
        )

        canvas_window = canvas.create_window(
            (0, 0),
            window=input_frame,
            anchor="nw"
        )

        canvas.bind(
            "<Configure>",
            lambda e:
            canvas.itemconfig(
                canvas_window,
                width=e.width
            )
        )

        canvas.configure(
            yscrollcommand=
            scrollbar.set
        )

        canvas.pack(
            side="left",
            fill="both",
            expand=True,
            padx=(15, 0)
        )

        scrollbar.pack(
            side="right",
            fill="y",
            pady=5
        )

        # --------------------------------------------------------
        # QUICK INPUT
        # --------------------------------------------------------

        quick = tk.Frame(
            input_frame,
            bg=self.colors["light_blue"],
            highlightbackground="#C9DFEF",
            highlightthickness=1
        )

        quick.pack(
            fill="x",
            padx=5,
            pady=(0, 12)
        )

        tk.Label(
            quick,
            text="Quick samples",
            font=("Segoe UI", 9, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["light_blue"]
        ).pack(
            side="left",
            padx=10,
            pady=8
        )

        ttk.Button(
            quick,
            text="Random",
            command=self.fill_random_sample
        ).pack(
            side="left",
            padx=3,
            pady=5
        )

        ttk.Button(
            quick,
            text="Observed Minimum",
            command=self.fill_best_case
        ).pack(
            side="left",
            padx=3,
            pady=5
        )

        ttk.Button(
            quick,
            text="Observed Maximum",
            command=self.fill_worst_case
        ).pack(
            side="left",
            padx=3,
            pady=5
        )

        # --------------------------------------------------------
        # INPUTS
        # --------------------------------------------------------

        self.entries = {}

        for index, feature in enumerate(
            self.feature_names
        ):

            row = tk.Frame(
                input_frame,
                bg=self.colors["card"]
            )

            row.pack(
                fill="x",
                padx=5,
                pady=4
            )

            tk.Label(
                row,
                text=feature,
                font=("Segoe UI", 9, "bold"),
                fg=self.colors["text"],
                bg=self.colors["card"],
                anchor="w",
                width=28
            ).pack(
                side="left"
            )

            stats = self.feature_stats[
                feature
            ]

            if feature in self.categorical_features:

                combo = ttk.Combobox(
                    row,
                    width=24,
                    state="readonly"
                )

                categories = (
                    stats["categories"]
                )

                combo["values"] = categories

                combo.set(
                    stats["mode"]
                )

                combo.pack(
                    side="left",
                    padx=6
                )

                self.entries[
                    feature
                ] = combo

            else:

                entry = ttk.Entry(
                    row,
                    width=18
                )

                entry.insert(
                    0,
                    f"{stats['mean']:.4f}"
                )

                entry.pack(
                    side="left",
                    padx=6
                )

                tk.Label(
                    row,
                    text=(
                        f"{stats['min']:.3f}  to  "
                        f"{stats['max']:.3f}"
                    ),
                    font=("Segoe UI", 8),
                    fg=self.colors["muted"],
                    bg=self.colors["card"]
                ).pack(
                    side="left"
                )

                self.entries[
                    feature
                ] = entry

        # --------------------------------------------------------
        # BUTTONS
        # --------------------------------------------------------

        button_frame = tk.Frame(
            input_frame,
            bg=self.colors["card"]
        )

        button_frame.pack(
            fill="x",
            padx=5,
            pady=18
        )

        self.predict_btn = ttk.Button(
            button_frame,
            text="PREDICT CHLORIDE",
            style="Accent.TButton",
            command=self.threaded_predict
        )

        self.predict_btn.pack(
            side="left",
            padx=3
        )

        ttk.Button(
            button_frame,
            text="Reset",
            command=self.clear_inputs
        ).pack(
            side="left",
            padx=5
        )

        # --------------------------------------------------------
        # RIGHT PANEL
        # --------------------------------------------------------

        right_card = tk.Frame(
            main,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        right_card.pack(
            side="right",
            fill="both",
            expand=True,
            padx=(8, 0)
        )

        tk.Label(
            right_card,
            text="Prediction Result",
            font=("Segoe UI", 15, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["card"]
        ).pack(
            anchor="w",
            padx=22,
            pady=(18, 0)
        )

        tk.Label(
            right_card,
            text="Estimated total chloride content",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["card"]
        ).pack(
            anchor="w",
            padx=22
        )

        # --------------------------------------------------------
        # RESULT CARD
        # --------------------------------------------------------

        result_card = tk.Frame(
            right_card,
            bg=self.colors["light_blue"],
            highlightbackground="#C9DFEF",
            highlightthickness=1
        )

        result_card.pack(
            fill="x",
            padx=22,
            pady=18
        )

        self.result_var = tk.StringVar(
            value="READY"
        )

        tk.Label(
            result_card,
            textvariable=self.result_var,
            font=("Segoe UI", 34, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["light_blue"]
        ).pack(
            pady=(20, 0)
        )

        tk.Label(
            result_card,
            text="% of binder mass",
            font=("Segoe UI", 11),
            fg=self.colors["muted"],
            bg=self.colors["light_blue"]
        ).pack(
            pady=(0, 18)
        )

        self.classification_var = (
            tk.StringVar(
                value=""
            )
        )

        self.classification_label = tk.Label(
            result_card,
            textvariable=
            self.classification_var,
            font=("Segoe UI", 13, "bold"),
            fg=self.colors["blue"],
            bg=self.colors["light_blue"]
        )

        self.classification_label.pack(
            pady=(0, 18)
        )

        # --------------------------------------------------------
        # ACTIONS
        # --------------------------------------------------------

        actions = tk.Frame(
            right_card,
            bg=self.colors["card"]
        )

        actions.pack(
            fill="x",
            padx=22,
            pady=5
        )

        ttk.Button(
            actions,
            text="Save Result",
            command=self.save_result
        ).pack(
            side="left",
            padx=(0, 5)
        )

        ttk.Button(
            actions,
            text="Copy",
            command=self.copy_to_clipboard
        ).pack(
            side="left",
            padx=5
        )

        ttk.Button(
            actions,
            text="Export History",
            command=self.export_history
        ).pack(
            side="left",
            padx=5
        )

        return tab

    # ============================================================
    # METRIC CARD
    # ============================================================

    def metric_card(
        self,
        parent,
        title,
        value,
        background,
        foreground
    ):

        card = tk.Frame(
            parent,
            bg=background,
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        tk.Label(
            card,
            text=title,
            font=("Segoe UI", 9, "bold"),
            fg=self.colors["muted"],
            bg=background
        ).pack(
            pady=(12, 0)
        )

        tk.Label(
            card,
            text=value,
            font=("Segoe UI", 17, "bold"),
            fg=foreground,
            bg=background
        ).pack(
            pady=(2, 12)
        )

        return card

    # ============================================================
    # ANALYSIS TAB
    # ============================================================

    def create_analysis_tab(self):

        tab = ttk.Frame(
            self.notebook
        )

        top = tk.Frame(
            tab,
            bg=self.colors["background"]
        )

        top.pack(
            fill="x",
            padx=15,
            pady=(15, 8)
        )

        tk.Label(
            top,
            text="Model Analysis",
            font=("Segoe UI", 17, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["background"]
        ).pack(
            side="left"
        )

        tk.Label(
            top,
            text="Independent test-set diagnostics",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["background"]
        ).pack(
            side="left",
            padx=15,
            pady=5
        )

        controls = tk.Frame(
            tab,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        controls.pack(
            fill="x",
            padx=15,
            pady=5
        )

        buttons = [

            (
                "Actual vs Predicted",
                self.plot_actual_vs_predicted
            ),

            (
                "Residual Analysis",
                self.plot_residual_analysis
            ),

            (
                "Correlation Matrix",
                self.plot_correlation_matrix
            )
        ]

        for text, command in buttons:

            ttk.Button(
                controls,
                text=text,
                command=command,
                width=24
            ).pack(
                side="left",
                padx=8,
                pady=10
            )

        self.analysis_frame = tk.Frame(
            tab,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        self.analysis_frame.pack(
            fill="both",
            expand=True,
            padx=15,
            pady=10
        )

        self.plot_actual_vs_predicted()

        return tab

    # ============================================================
    # HISTORY TAB
    # ============================================================

    def create_history_tab(self):

        tab = ttk.Frame(
            self.notebook
        )

        top = tk.Frame(
            tab,
            bg=self.colors["background"]
        )

        top.pack(
            fill="x",
            padx=15,
            pady=15
        )

        tk.Label(
            top,
            text="Prediction History",
            font=("Segoe UI", 17, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["background"]
        ).pack(
            side="left"
        )

        self.history_stats_var = (
            tk.StringVar(
                value="No predictions yet"
            )
        )

        tk.Label(
            top,
            textvariable=
            self.history_stats_var,
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["background"]
        ).pack(
            side="right"
        )

        controls = tk.Frame(
            tab,
            bg=self.colors["background"]
        )

        controls.pack(
            fill="x",
            padx=15
        )

        ttk.Button(
            controls,
            text="Refresh",
            command=self.update_history_display
        ).pack(
            side="left",
            padx=4
        )

        ttk.Button(
            controls,
            text="Export CSV / Excel",
            command=self.export_history
        ).pack(
            side="left",
            padx=4
        )

        ttk.Button(
            controls,
            text="Clear History",
            command=self.clear_history
        ).pack(
            side="left",
            padx=4
        )

        # --------------------------------------------------------
        # LIST
        # --------------------------------------------------------

        card = tk.Frame(
            tab,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        card.pack(
            fill="both",
            expand=True,
            padx=15,
            pady=12
        )

        self.history_listbox = tk.Listbox(
            card,
            font=("Consolas", 10),
            bg=self.colors["card"],
            fg=self.colors["text"],
            selectbackground=self.colors["blue"],
            selectforeground=self.colors["white"],
            relief="flat",
            borderwidth=0
        )

        self.history_listbox.pack(
            side="left",
            fill="both",
            expand=True,
            padx=10,
            pady=10
        )

        scrollbar = ttk.Scrollbar(
            card,
            orient="vertical",
            command=
            self.history_listbox.yview
        )

        scrollbar.pack(
            side="right",
            fill="y",
            pady=10
        )

        self.history_listbox.configure(
            yscrollcommand=
            scrollbar.set
        )

        self.history_listbox.bind(
            "<Double-Button-1>",
            self.load_history_entry
        )

        return tab

    # ============================================================
    # INPUT HELPERS
    # ============================================================

    def fill_random_sample(self):

        index = np.random.randint(
            len(self.X_train_raw)
        )

        sample = self.X_train_raw.iloc[
            index
        ]

        self.fill_raw_sample(
            sample
        )

    def fill_raw_sample(
        self,
        sample
    ):

        for feature in self.feature_names:

            if feature not in sample:

                continue

            widget = self.entries[
                feature
            ]

            value = sample[
                feature
            ]

            if feature in self.categorical_features:

                widget.set(
                    str(value)
                )

            else:

                widget.delete(
                    0,
                    tk.END
                )

                widget.insert(
                    0,
                    str(value)
                )

    def fill_best_case(self):

        idx = self.y_train.idxmin()

        sample = (
            self.X_train_raw.loc[idx]
        )

        self.fill_raw_sample(
            sample
        )

    def fill_worst_case(self):

        idx = self.y_train.idxmax()

        sample = (
            self.X_train_raw.loc[idx]
        )

        self.fill_raw_sample(
            sample
        )

    # ============================================================
    # BUILD PREDICTION DATA
    # ============================================================

    def build_prediction_dataframe(self):

        data = {}

        for feature in self.feature_names:

            widget = self.entries[
                feature
            ]

            value = widget.get().strip()

            # ----------------------------------------------------
            # CATEGORICAL
            # ----------------------------------------------------

            if feature in self.categorical_features:

                mapping = (
                    self.categorical_maps[
                        feature
                    ]
                )

                if value not in mapping:

                    raise ValueError(
                        f"Unknown category for "
                        f"'{feature}': {value}"
                    )

                data[feature] = (
                    mapping[value]
                )

            # ----------------------------------------------------
            # NUMERIC
            # ----------------------------------------------------

            else:

                try:

                    numeric_value = float(
                        value
                    )

                except ValueError:

                    raise ValueError(
                        f"Invalid numeric value "
                        f"for '{feature}'."
                    )

                if not np.isfinite(
                    numeric_value
                ):

                    raise ValueError(
                        f"Invalid value for "
                        f"'{feature}'."
                    )

                data[feature] = (
                    numeric_value
                )

        return pd.DataFrame(
            [data],
            columns=self.feature_names
        )

    # ============================================================
    # PREDICTION
    # ============================================================

    def threaded_predict(self):

        self.predict_btn.config(
            state="disabled",
            text="PREDICTING..."
        )

        self.status_var.set(
            "Generating TabPFN prediction..."
        )

        thread = threading.Thread(
            target=self.predict_ingress,
            daemon=True
        )

        thread.start()

    def predict_ingress(self):

        try:

            input_data = (
                self.build_prediction_dataframe()
            )

            start = time.time()

            prediction = float(
                self.model.predict(
                    input_data
                )[0]
            )

            elapsed = (
                time.time() - start
            )

            self.root.after(
                0,
                lambda:
                self.display_result(
                    prediction,
                    input_data,
                    elapsed
                )
            )

        except Exception as e:

            self.root.after(
                0,
                lambda:
                messagebox.showerror(
                    "Prediction Error",
                    str(e)
                )
            )

        finally:

            self.root.after(
                0,
                self.enable_predict_button
            )

    def enable_predict_button(self):

        self.predict_btn.config(
            state="normal",
            text="PREDICT CHLORIDE"
        )

        self.status_var.set(
            "Model ready"
        )

    # ============================================================
    # CLASSIFICATION
    # ============================================================

    def classify_chloride(
        self,
        chloride
    ):

        if chloride < 0.2:

            return (
                "VERY LOW",
                self.colors["green"]
            )

        if chloride < 0.5:

            return (
                "LOW",
                self.colors["blue"]
            )

        if chloride < 1.0:

            return (
                "MODERATE",
                self.colors["orange"]
            )

        if chloride < 2.0:

            return (
                "HIGH",
                "#B7791F"
            )

        return (
            "VERY HIGH",
            self.colors["red"]
        )

    # ============================================================
    # DISPLAY RESULT
    # ============================================================

    def display_result(
        self,
        prediction,
        input_data,
        elapsed
    ):

        self.result_var.set(
            f"{prediction:.4f}%"
        )

        classification, color = (
            self.classify_chloride(
                prediction
            )
        )

        self.classification_var.set(
            classification
        )

        self.classification_label.config(
            fg=color
        )

        self.status_var.set(
            f"Prediction completed in "
            f"{elapsed:.3f} s"
        )

        self.save_to_history(
            prediction,
            classification,
            input_data
        )

    # ============================================================
    # HISTORY
    # ============================================================

    def save_to_history(
        self,
        prediction,
        classification,
        input_data
    ):

        entry = {

            "timestamp":
                time.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),

            "chloride":
                float(prediction),

            "classification":
                classification
        }

        for feature in self.feature_names:

            value = input_data[
                feature
            ].iloc[0]

            entry[feature] = (
                float(value)
                if np.isscalar(value)
                else str(value)
            )

        self.prediction_history.append(
            entry
        )

        self.save_history()

        self.update_history_display()

    def update_history_display(self):

        self.history_listbox.delete(
            0,
            tk.END
        )

        for entry in self.prediction_history:

            text = (
                f"{entry['timestamp']}   |   "
                f"{entry['chloride']:.4f}%   |   "
                f"{entry['classification']}"
            )

            self.history_listbox.insert(
                tk.END,
                text
            )

        if self.prediction_history:

            values = [
                x["chloride"]
                for x in self.prediction_history
            ]

            self.history_stats_var.set(
                f"{len(values)} predictions   |   "
                f"Mean {np.mean(values):.4f}%   |   "
                f"Min {np.min(values):.4f}%   |   "
                f"Max {np.max(values):.4f}%"
            )

        else:

            self.history_stats_var.set(
                "No predictions yet"
            )

    def load_history_entry(
        self,
        event=None
    ):

        selection = (
            self.history_listbox
            .curselection()
        )

        if not selection:

            return

        index = selection[0]

        if index >= len(
            self.prediction_history
        ):

            return

        entry = (
            self.prediction_history[
                index
            ]
        )

        for feature in self.feature_names:

            if feature not in entry:

                continue

            widget = self.entries[
                feature
            ]

            value = str(
                entry[feature]
            )

            if feature in self.categorical_features:

                widget.set(
                    value
                )

            else:

                widget.delete(
                    0,
                    tk.END
                )

                widget.insert(
                    0,
                    value
                )

        self.result_var.set(
            f"{entry['chloride']:.4f}%"
        )

        self.classification_var.set(
            entry["classification"]
        )

    def save_history(self):

        try:

            with open(
                self.history_file,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    self.prediction_history,
                    f,
                    indent=4
                )

        except Exception as e:

            print(
                f"History save error: {e}"
            )

    def load_history(self):

        if not os.path.exists(
            self.history_file
        ):

            return

        try:

            with open(
                self.history_file,
                "r",
                encoding="utf-8"
            ) as f:

                self.prediction_history = (
                    json.load(f)
                )

            self.update_history_display()

        except Exception:

            self.prediction_history = []

    def clear_history(self):

        if not self.prediction_history:

            return

        answer = messagebox.askyesno(
            "Clear History",
            "Delete all prediction history?"
        )

        if answer:

            self.prediction_history = []

            self.save_history()

            self.update_history_display()

    def export_history(self):

        if not self.prediction_history:

            messagebox.showinfo(
                "No Data",
                "No prediction history exists."
            )

            return

        filename = (
            filedialog.asksaveasfilename(
                defaultextension=".csv",
                filetypes=[
                    (
                        "CSV files",
                        "*.csv"
                    ),
                    (
                        "Excel files",
                        "*.xlsx"
                    )
                ]
            )
        )

        if not filename:

            return

        df = pd.DataFrame(
            self.prediction_history
        )

        if filename.lower().endswith(
            ".xlsx"
        ):

            df.to_excel(
                filename,
                index=False
            )

        else:

            df.to_csv(
                filename,
                index=False
            )

        messagebox.showinfo(
            "Export Complete",
            "Prediction history exported successfully."
        )

    # ============================================================
    # CLEAR INPUTS
    # ============================================================

    def clear_inputs(self):

        for feature in self.feature_names:

            widget = self.entries[
                feature
            ]

            stats = self.feature_stats[
                feature
            ]

            if feature in self.categorical_features:

                widget.set(
                    stats["mode"]
                )

            else:

                widget.delete(
                    0,
                    tk.END
                )

                widget.insert(
                    0,
                    f"{stats['mean']:.4f}"
                )

        self.result_var.set(
            "READY"
        )

        self.classification_var.set(
            ""
        )

        self.status_var.set(
            "Model ready"
        )

    # ============================================================
    # SAVE RESULT
    # ============================================================

    def save_result(self):

        filename = (
            filedialog.asksaveasfilename(
                defaultextension=".txt",
                filetypes=[
                    (
                        "Text files",
                        "*.txt"
                    )
                ],
                initialfile=
                "chloride_prediction.txt"
            )
        )

        if not filename:

            return

        text = (
            "CHLORIDE INGRESS PREDICTION\n"
            "====================================\n\n"
            f"Predicted chloride: "
            f"{self.result_var.get()}\n"
            f"Classification: "
            f"{self.classification_var.get()}\n\n"
            "INPUT PARAMETERS\n"
            "------------------------------------\n"
        )

        for feature in self.feature_names:

            text += (
                f"{feature}: "
                f"{self.entries[feature].get()}\n"
            )

        text += (
            "\nMODEL PERFORMANCE\n"
            "------------------------------------\n"
            f"Model: TabPFN Regressor\n"
            f"Test R²: {self.test_r2:.4f}\n"
            f"Test RMSE: {self.test_rmse:.6f}\n"
            f"Test MAE: {self.test_mae:.6f}\n"
            f"Test MAPE: {self.test_mape:.2f}%\n"
        )

        with open(
            filename,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(text)

        messagebox.showinfo(
            "Saved",
            "Prediction saved successfully."
        )

    # ============================================================
    # COPY
    # ============================================================

    def copy_to_clipboard(self):

        text = (
            f"Chloride ingress: "
            f"{self.result_var.get()} | "
            f"{self.classification_var.get()}"
        )

        self.root.clipboard_clear()

        self.root.clipboard_append(
            text
        )

        self.root.update()

    # ============================================================
    # ANALYSIS FRAME
    # ============================================================

    def clear_analysis_frame(self):

        for widget in (
            self.analysis_frame
            .winfo_children()
        ):

            widget.destroy()

    def show_figure(
        self,
        fig
    ):

        self.clear_analysis_frame()

        canvas = FigureCanvasTkAgg(
            fig,
            self.analysis_frame
        )

        canvas.draw()

        canvas.get_tk_widget().pack(
            fill="both",
            expand=True,
            padx=8,
            pady=8
        )

        self.current_plot = fig

    # ============================================================
    # ACTUAL VS PREDICTED
    # ============================================================

    def plot_actual_vs_predicted(self):

        fig = Figure(
            figsize=(10, 7),
            dpi=100,
            facecolor=self.colors["card"]
        )

        ax = fig.add_subplot(111)

        ax.set_facecolor(
            self.colors["card"]
        )

        ax.scatter(
            self.y_test,
            self.y_test_pred,
            alpha=0.75,
            s=60
        )

        minimum = min(
            self.y_test.min(),
            self.y_test_pred.min()
        )

        maximum = max(
            self.y_test.max(),
            self.y_test_pred.max()
        )

        ax.plot(
            [minimum, maximum],
            [minimum, maximum],
            linestyle="--",
            linewidth=2
        )

        ax.set_xlabel(
            "Actual Chloride (% of binder mass)",
            fontsize=11
        )

        ax.set_ylabel(
            "Predicted Chloride (% of binder mass)",
            fontsize=11
        )

        ax.set_title(
            "Actual vs Predicted",
            fontsize=15,
            fontweight="bold"
        )

        ax.grid(
            True,
            alpha=0.2
        )

        text = (
            f"R² = {self.test_r2:.4f}\n"
            f"RMSE = {self.test_rmse:.5f}\n"
            f"MAE = {self.test_mae:.5f}\n"
            f"n = {len(self.X_test)}"
        )

        ax.text(
            0.04,
            0.96,
            text,
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=10,
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor="white",
                alpha=0.9
            )
        )

        fig.tight_layout()

        self.show_figure(
            fig
        )

    # ============================================================
    # RESIDUAL ANALYSIS
    # ============================================================

    def plot_residual_analysis(self):

        residuals = (
            self.y_test.values
            -
            self.y_test_pred
        )

        fig = Figure(
            figsize=(10, 7),
            dpi=100,
            facecolor=self.colors["card"]
        )

        ax = fig.add_subplot(111)

        ax.set_facecolor(
            self.colors["card"]
        )

        ax.scatter(
            self.y_test_pred,
            residuals,
            alpha=0.75,
            s=60
        )

        ax.axhline(
            0,
            linestyle="--",
            linewidth=2
        )

        ax.set_xlabel(
            "Predicted Chloride (%)",
            fontsize=11
        )

        ax.set_ylabel(
            "Residual",
            fontsize=11
        )

        ax.set_title(
            "Residual Analysis",
            fontsize=15,
            fontweight="bold"
        )

        ax.grid(
            True,
            alpha=0.2
        )

        fig.tight_layout()

        self.show_figure(
            fig
        )

    # ============================================================
    # CORRELATION MATRIX
    # ============================================================

    def plot_correlation_matrix(self):

        numeric_data = (
            self.X_full
            .select_dtypes(
                include=np.number
            )
            .copy()
        )

        numeric_data[
            self.target_name
        ] = self.y_full.values

        if numeric_data.shape[1] < 2:

            messagebox.showinfo(
                "Correlation",
                "Not enough numerical variables."
            )

            return

        corr = (
            numeric_data
            .corr()
        )

        fig = Figure(
            figsize=(11, 9),
            dpi=100,
            facecolor=self.colors["card"]
        )

        ax = fig.add_subplot(111)

        image = ax.imshow(
            corr,
            vmin=-1,
            vmax=1,
            aspect="auto"
        )

        labels = [
            str(x)[:18]
            for x in corr.columns
        ]

        ax.set_xticks(
            np.arange(
                len(labels)
            )
        )

        ax.set_yticks(
            np.arange(
                len(labels)
            )
        )

        ax.set_xticklabels(
            labels,
            rotation=45,
            ha="right",
            fontsize=8
        )

        ax.set_yticklabels(
            labels,
            fontsize=8
        )

        for i in range(
            len(labels)
        ):

            for j in range(
                len(labels)
            ):

                ax.text(
                    j,
                    i,
                    f"{corr.iloc[i, j]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7
                )

        fig.colorbar(
            image,
            ax=ax,
            fraction=0.046,
            pad=0.04
        )

        ax.set_title(
            "Correlation Matrix",
            fontsize=15,
            fontweight="bold"
        )

        fig.tight_layout()

        self.show_figure(
            fig
        )

    # ============================================================
    # RUN
    # ============================================================

    def run(self):

        print("\n" + "=" * 70)
        print(
            "CHLORIDE INGRESS PREDICTOR"
        )
        print("=" * 70)

        print(
            f"Target: {self.target_name}"
        )

        print(
            f"Features: {len(self.feature_names)}"
        )

        print(
            f"Training samples: "
            f"{len(self.X_train)}"
        )

        print(
            f"Test samples: "
            f"{len(self.X_test)}"
        )

        print(
            f"Test R²: "
            f"{self.test_r2:.4f}"
        )

        print(
            f"Test RMSE: "
            f"{self.test_rmse:.6f}"
        )

        print(
            f"Test MAE: "
            f"{self.test_mae:.6f}"
        )

        print("=" * 70)
        print("GUI READY")
        print("=" * 70)

        self.root.mainloop()

    # ============================================================
    # CLOSE
    # ============================================================

    def on_closing(self):

        self.save_history()

        self.root.destroy()


# ================================================================
# MAIN
# ================================================================

if __name__ == "__main__":

    try:

        start_time = time.time()

        print(
            "\nInitializing Chloride "
            "Ingress Predictor..."
        )

        app = ChlorideIngressPredictor()

        print(
            f"\nApplication initialization: "
            f"{time.time() - start_time:.2f} s"
        )

        app.run()

    except Exception as e:

        print(
            "\nAPPLICATION ERROR"
        )

        print(
            str(e)
        )

        import traceback

        traceback.print_exc()

        try:

            messagebox.showerror(
                "Application Error",
                str(e)
            )

        except Exception:

            pass


Initializing Chloride Ingress Predictor...

LOADING CHLORIDE INGRESS DATA
CSV encoding: cp1252
Dataset shape: (918, 16)
Target: Total chloride, % of binder mass
Features: 15
Categorical features: 0
Preprocessing completed in 1.66 s
GPU not detected. CPU mode.

Loading saved TabPFN model...
Saved model loaded.

Application initialization: 81.53 s

CHLORIDE INGRESS PREDICTOR
Target: Total chloride, % of binder mass
Features: 15
Training samples: 734
Test samples: 184
Test R²: 0.9516
Test RMSE: 0.288409
Test MAE: 0.155343
GUI READY
